# Entrenamiento del modelo LSTM
Pipeline: `10_empaquetar_dataset.py` → **este notebook**

In [ ]:
import os
SEED = 42
os.environ['PYTHONHASHSEED']         = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['CUDA_VISIBLE_DEVICES']   = ''   # fuerza CPU para maximo determinismo
os.environ['TF_ENABLE_ONEDNN_OPTS']  = '0'  # evita reduccion multi-hilo no determinista

import sys
import random
import numpy as np
import pandas as pd
import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
from pathlib import Path
from tensorflow.keras.utils import to_categorical

sys.path.insert(0, str(Path('..').resolve()))
from utils.config import DATA_DIR, DATASET_V2_DIR
from utils.model import construir_lstm, callbacks_entrenamiento

# Semillas fijas para reproducibilidad
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SALIDA_DIR = os.path.join(DATA_DIR, 'modelo', 'resultados')
CHECKPOINT = os.path.join(SALIDA_DIR, 'mejor_modelo_v2.keras')
os.makedirs(SALIDA_DIR, exist_ok=True)
print('Librerías cargadas correctamente')

In [2]:
# Cargar datos empaquetados
X_train = np.load(os.path.join(DATASET_V2_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATASET_V2_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(DATASET_V2_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(DATASET_V2_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(DATASET_V2_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(DATASET_V2_DIR, 'y_test.npy'))
clases  = np.load(os.path.join(DATASET_V2_DIR, 'clases.npy'), allow_pickle=True)

n_clases = len(clases)
print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')
print(f'Clases: {n_clases}')

Train : (1079, 60, 225)
Val   : (35, 60, 225)
Test  : (35, 60, 225)
Clases: 35


## Arquitectura del modelo
Captura la salida de esta celda para incluirla en la tesis (§2.10).

In [3]:
model = construir_lstm(n_clases=n_clases, lr=0.001, dropout=0.5,
                       regularizacion=0.005, capa_densa=False)
model.summary()

Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 lstm (LSTM)                 (None, 60, 64)            74240     


 batch_normalization (Batch  (None, 60, 64)            256       


...

 dense (Dense)               (None, 35)                1155      


Total params: 88195 (344.51 KB)


Trainable params: 88003 (343.76 KB)


Non-trainable params: 192 (768.00 Byte)


_________________________________________________________________


## Entrenamiento

In [4]:
history = model.fit(
    X_train, to_categorical(y_train, n_clases),
    validation_data=(X_val, to_categorical(y_val, n_clases)),
    epochs=100,
    batch_size=32,
    callbacks=callbacks_entrenamiento(CHECKPOINT, patience_stop=20, patience_lr=7),
    verbose=1,
)

Epoch 1/100


 1/34 [..............................] - ETA: 2:15 - loss: 6.0437 - accuracy: 0.0312

 2/34 [>.............................] - ETA: 3s - loss: 5.9837 - accuracy: 0.0156  

 3/34 [=>............................] - ETA: 2s - loss: 5.9939 - accuracy: 0.0104

 4/34 [==>...........................] - ETA: 2s - loss: 5.9932 - accuracy: 0.0156

...

32/34 [===========================>..] - ETA: 0s - loss: 0.4134 - accuracy: 0.9854

33/34 [============================>.] - ETA: 0s - loss: 0.4113 - accuracy: 0.9858

Restoring model weights from the end of the best epoch: 30.


34/34 [==============================] - 2s 47ms/step - loss: 0.4119 - accuracy: 0.9861 - val_loss: 3.4221 - val_accuracy: 0.3143 - lr: 1.2500e-04


Epoch 50: early stopping


## Resultados finales

In [5]:
_, acc   = model.evaluate(X_test, to_categorical(y_test, n_clases), verbose=0)
val_acc  = max(history.history['val_accuracy'])
tr_acc   = max(history.history['accuracy'])
epocas   = len(history.history['accuracy'])

print(f'Test accuracy    : {acc*100:.1f}%')
print(f'Val accuracy max : {val_acc*100:.1f}%')
print(f'Sobreajuste      : {(tr_acc - val_acc)*100:.1f} pp')
print(f'Épocas ejecutadas: {epocas}')

Test accuracy    : 48.6%
Val accuracy max : 42.9%
Sobreajuste      : 56.5 pp
Épocas ejecutadas: 50


## Guardar modelo e historial

In [ ]:
model.save(os.path.join(SALIDA_DIR, 'modelo_lstm_final_v2.keras'))
np.save(os.path.join(SALIDA_DIR, 'historial_v2.npy'), history.history)
print(f'Modelos guardados en: {SALIDA_DIR}')

## Curvas de aprendizaje

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validación')
ax1.set_title('Exactitud por época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Exactitud')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validación')
ax2.set_title('Pérdida por época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SALIDA_DIR, 'curvas_aprendizaje.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Curvas guardadas en curvas_aprendizaje.png')